#### Goal
- To Achive Data Model result by considering All the source systems where W&R Clients are available and GCDS source

#### author
- Abhishek.R.Jaiswal@rabobank.com

##### flow of logic
- Take W&R clients data from different source systems
- Take GCDS data
- Join on GCDSId to further derive required output

#### Expected output
  - Column List is added at the end of this notebook 


#####Read Files from GDP

In [0]:
import os
import pandas as pd
app_reg_app_id = os.environ['APP_REG_APP_ID']
ReadStorage = os.environ['GDP_STORAGE_NAME']
TenantId = os.environ['TENANT_ID']
GIC_ReadStorage = os.environ['GDP_SA_STORAGE_NAME']
environment=os.environ['ENV']
radar_datamodel_version_number=3
service_credential = dbutils.secrets.get(scope="connectedsecrets", key = f"appreg-{app_reg_app_id}")

In [0]:
from RadarUtils import *

In [0]:
party_dataobject='Internal_SystemComparison'

In [0]:
spark.conf.set("fs.azure.account.auth.type."+ReadStorage+".dfs.core.windows.net", "OAuth") 
spark.conf.set("fs.azure.account.oauth.provider.type."+ReadStorage+".dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider") 
spark.conf.set("fs.azure.account.oauth2.client.id."+ReadStorage+".dfs.core.windows.net", ""+app_reg_app_id+"") 
spark.conf.set("fs.azure.account.oauth2.client.secret."+ReadStorage+".dfs.core.windows.net", service_credential) 
spark.conf.set("fs.azure.account.oauth2.client.endpoint."+ReadStorage+".dfs.core.windows.net", "https://login.microsoftonline.com/"+TenantId+"/oauth2/token") 

In [0]:
# print list of strings for loading spark dfs from GDP
gcds_df = pd.DataFrame({'definedDatasetname':[
'client_Client',
'client_ClientOwnersProduct',
# 'client_AttributeExtensionsAttribute',
'client_ClientOwnersLocal',
'client_KeyStoreKey',
'client_OnboardedLocations',
'client_PartyRole',
'client_RMA' ,
'client_PartytoPartyRelationship',
'client_Products'
]})

# Create TempView for each loading table
for index, row in gcds_df.iterrows():
    Read_GDP_Defined_DataObjects(Source='GCDS', Dataobject=row.definedDatasetname)

In [0]:
# print list of strings for loading spark dfs from GDP
load_df =[
'party_case_client_details',
'party_client_structure_GUI',
'party_RelatedPartyParentAddresses',
'party_client',
'party_trade_name',
'party_Alias',
'party_AllPartyDetails' 
]

# Create TempView for each loading table
for Object in load_df:
    Read_GDP_Defined_DataObjects(Source='GCOB', Dataobject=Object, path_prefix='CaseService')

#####GCDS - GCOB Attribute Comparison

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW gcdsRawData AS
SELECT DISTINCT
  t1.gcid
  , t1.keystore_type
  , t1.KeyStore_value AS gcobid
  , t1.status
  , t2.Full_legal_name AS gcdsClientName
  , t2.ResidentialAddress_Country AS gcdsResidentialAddressCountry
  , t2.Principal_address_country AS gcdsPrincipalAddressCountry
  , t2.Party_type AS gcdsPartyType
  , t2.`CDD-entitytype` AS gcdsCddEntitytype
  , t2.`CDD-next_reviewdate` AS gcdsCDDNextReviewdate
  , t2.`CDD-risk_rating` AS gcdsCDDRiskRating
  , t2.`Global_CO-name` AS gcdsGlobalCOname
  , t2.`Global_CO-location` AS gcdsGlobalCOlocation
  , t2.`Global_CO-email` AS globalCOemail
  , t2.`CO-businessline_description` AS gcdsCObusinesslinedescription
  , t2.`Global_CO-Serviced_by` AS gcdsGlobalCOServicedBy
  , t2.`RM-name` AS gcdsRMName
  , t2.`RM-location` AS gcdsRMLocation
  , t2.`RM-email` AS gcdsRMEmail
  , t2.`RM-businessline_description` AS gcdsRMBusinesslineDescription
  , t2.`RM-_Serviced_by` AS gcdsRMServicedBy
  , t2.`CO-email` AS gcdsGCOemail
  , t2.Registered_address_country AS gcdsRegisteredCountry
  , t2.`CDD-risk_rating` AS gcdsCDDRating
  , t3.Life_cycle_status AS gcdsClientlifecyclestatus
  , t3.party_role AS gcdsPartyRole
  , t2.Customer_ambition AS clientStrategy
  , t2.Registered_address_streetLine1
  , t2.Registered_address_house_nr
  , t2.Registered_address_zipcode
  , t2.Registered_address_city
  , t2.Registered_address_region
  , t2.Principal_address_country_code
  , CONCAT(t2.Registered_address_zipcode, t2.Registered_address_house_nr) AS gcdsAddress
  , CONCAT(t2.Registered_address_city, t2.Registered_address_region) AS gcdsCityRegion
  , t4.`Relationship-Type` AS relationshipType

FROM client_KeyStoreKey t1
LEFT JOIN client_client t2 ON t1.gcid = t2.gcid
LEFT JOIN client_PartyRole t3 ON t1.gcid = t3.gcid
LEFT JOIN client_PartytoPartyRelationship t4 ON t1.gcid = t4.gcid
WHERE t1.keystore_type IN ('GCOBID')
  AND t1.status = 'Active'
  AND t3.party_role = 'Customer'
  --AND t4.`Relationship-Type` <> 'Has Fund Manager'

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW sggGcobData AS
select distinct 
  t1.uniquegcobid,
  t1.FullLegalName,
  t1.clientlifecyclename,
  t2.IncorporationNumber,
  CONCAT(REPLACE(t2.RegisteredPostalCode, ' ', ''), t2.RegisteredNumber) AS gcobAddress,
  CONCAT(t2.RegisteredCity, t2.RegisteredRegion) AS gcobCityRegion,
  t2.RegisteredCountryIsoCode,
  t1.GlobalClientOwner,
  t1.GlobalClientOwnerLocation,
  t1.nextreviewdate,
  t2.ValidatedRiskLevel,
  t1.BusinessLineName,
  t1.GlobalKYCPortfolioNew,
  t1.SectorTeam,
  t1.KYCGroup,
  t1.GlobalReportingRegion,
  t1.ReviewLocation, 
  t3.ContactPersonEmailAddress, 
  t3.ContactPersonTelephoneNumber
from radar.clients as t1
left join radar.cases as t2 on t1.SourceClient = t2.SourceClient
left join party_case_client_details as t3 on t1.SourceClient = t3.SourceClient

In [0]:
%sql
/*
Description for the values
0: No Match
1: String Matching
2: Empty or Null
3: Special characters removed then string match
*/

CREATE OR REPLACE TEMPORARY VIEW GcobGcdsClientComparison AS
SELECT DISTINCT
  CONCAT(t5.UniqueGcobId, '_', t1.gcid) AS sggUniqueIdentifier
  , t1.gcid 
  , t5.UniqueGcobId
  , t1.gcdsPartyRole

  -- NAME COMPARISON
  , CASE
      WHEN LOWER(TRIM(t1.gcdsClientName)) = LOWER(TRIM(t5.FullLegalName)) THEN 1 -- string match
      WHEN LOWER(TRIM(
        REGEXP_REPLACE(
          REGEXP_REPLACE(
            REGEXP_REPLACE(
              REGEXP_REPLACE(
                REGEXP_REPLACE(
                  TRANSLATE(t1.gcdsClientName, 
                    'áàäâãåéèëêíìïîóòöôõúùüûçñÁÀÄÂÃÅÉÈËÊÍÌÏÎÓÒÖÔÕÚÙÜÛÇÑ', 
                    'aaaaaaeeeeiiiiooooouuuucnAAAAAAEEEEIIIIOOOOOUUUUCN'
                  ), 
                  'DLL', 'De Lage Landen'
                ), 
                'DAC', 'Designated Activity Company'
              ), 
              '(?i)\\bLtd\\.?', 'Limited'
            ), 
            '(?i)\\bLtda\\.?', 'Limitada'
          ),
          '[^a-zA-Z0-9]', ''
        )
      )) = LOWER(TRIM(
        REGEXP_REPLACE(
          REGEXP_REPLACE(
            REGEXP_REPLACE(
              REGEXP_REPLACE(
                REGEXP_REPLACE(
                  TRANSLATE(t5.FullLegalName, 
                    'áàäâãåéèëêíìïîóòöôõúùüûçñÁÀÄÂÃÅÉÈËÊÍÌÏÎÓÒÖÔÕÚÙÜÛÇÑ', 
                    'aaaaaaeeeeiiiiooooouuuucnAAAAAAEEEEIIIIOOOOOUUUUCN'
                  ), 
                  'DLL', 'De Lage Landen'
                ), 
                'DAC', 'Designated Activity Company'
              ), 
              '(?i)\\bLtd\\.?', 'Limited'
            ), 
            '(?i)\\bLtda\\.?', 'Limitada'
          ),
          '[^a-zA-Z0-9]', ''
        )
      )) THEN 3 -- special characters removed, string match
      WHEN t1.gcdsClientName IS NULL OR t5.FullLegalName IS NULL OR TRIM(t1.gcdsClientName) = '' OR TRIM(t5.FullLegalName) = '' THEN 2 -- empty or null
      ELSE 0 -- no match
    END AS gcobGcdsClientNameMatch



  -- Client lifecycle status comparison
  , CASE 
      WHEN LOWER(TRIM(t1.gcdsClientlifecyclestatus)) = 'client' AND LOWER(TRIM(t5.clientlifecyclename)) = 'client' THEN 1  -- string match
      WHEN LOWER(TRIM(t1.gcdsClientlifecyclestatus)) = 'former client' AND LOWER(TRIM(t5.clientlifecyclename)) = 'formerclient' THEN 1 -- string match
      WHEN LOWER(TRIM(t1.gcdsClientlifecyclestatus)) = 'prospect' AND LOWER(TRIM(t5.clientlifecyclename)) = 'prospect' THEN 1 -- string match
      WHEN t1.gcdsClientlifecyclestatus IS NULL OR t5.clientlifecyclename IS NULL OR TRIM(t1.gcdsClientlifecyclestatus) = '' OR TRIM(t5.clientlifecyclename) = '' THEN 2 -- empty or null
      ELSE 0 -- no match
    END AS gcobGcdsClientLifeCycleStatusMatch


  -- GCOB and GCDS address match
  , CASE
      WHEN LOWER(TRIM(t1.gcdsAddress)) = LOWER(TRIM(t5.gcobAddress)) THEN 1 -- string match
      WHEN LOWER(TRIM(
          REGEXP_REPLACE(
            TRANSLATE(t1.gcdsAddress, 
              'áàäâãåéèëêíìïîóòöôõúùüûçñÁÀÄÂÃÅÉÈËÊÍÌÏÎÓÒÖÔÕÚÙÜÛÇÑ', 
              'aaaaaaeeeeiiiiooooouuuucnAAAAAAEEEEIIIIOOOOOUUUUCN'
            ), 
            '[^a-zA-Z0-9]', ''
          )
        )) = LOWER(TRIM(
          REGEXP_REPLACE(
            TRANSLATE(t5.gcobAddress, 
              'áàäâãåéèëêíìïîóòöôõúùüûçñÁÀÄÂÃÅÉÈËÊÍÌÏÎÓÒÖÔÕÚÙÜÛÇÑ', 
              'aaaaaaeeeeiiiiooooouuuucnAAAAAAEEEEIIIIOOOOOUUUUCN'
            ), 
            '[^a-zA-Z0-9]', ''
          )
        )) THEN 3 -- special characters removed, string match
      WHEN t1.gcdsAddress IS NULL OR t5.gcobAddress IS NULL OR TRIM(t1.gcdsAddress) = '' OR TRIM(t5.gcobAddress) = '' THEN 2 -- empty or null
      ELSE 0 -- no match
    END AS gcdsGcobAddressMatch


  -- GCOB and GCDS City and Region match
  , CASE
      WHEN LOWER(TRIM(gcdsCityRegion)) = LOWER(TRIM(gcobCityRegion)) THEN 1 -- string match
      WHEN LOWER(TRIM(
        REGEXP_REPLACE(
          TRANSLATE(gcdsCityRegion, 
            'áàäâãåéèëêíìïîóòöôõúùüûçñÁÀÄÂÃÅÉÈËÊÍÌÏÎÓÒÖÔÕÚÙÜÛÇÑ', 
            'aaaaaaeeeeiiiiooooouuuucnAAAAAAEEEEIIIIOOOOOUUUUCN'
          ), 
          '[^a-zA-Z0-9]', ''
        )
      )) = LOWER(TRIM(
        REGEXP_REPLACE(
          TRANSLATE(gcobCityRegion, 
            'áàäâãåéèëêíìïîóòöôõúùüûçñÁÀÄÂÃÅÉÈËÊÍÌÏÎÓÒÖÔÕÚÙÜÛÇÑ', 
            'aaaaaaeeeeiiiiooooouuuucnAAAAAAEEEEIIIIOOOOOUUUUCN'
          ), 
          '[^a-zA-Z0-9]', ''
        )
      )) THEN 3 -- special characters removed, string match
      WHEN gcdsCityRegion IS NULL OR gcobCityRegion IS NULL OR TRIM(gcdsCityRegion) = '' OR TRIM(gcobCityRegion) = '' THEN 2 -- empty or null
      ELSE 0 -- no match
    END AS gcdsGcobCityRegionMatch


  -- GCOB & GCDS Country of registration
  , CASE
      WHEN LOWER(TRIM(t5.RegisteredCountryIsoCode)) = LOWER(TRIM(t1.Principal_address_country_code)) THEN 1 -- string match
      WHEN LOWER(TRIM(
        REGEXP_REPLACE(
          TRANSLATE(t5.RegisteredCountryIsoCode, 
            'áàäâãåéèëêíìïîóòöôõúùüûçñÁÀÄÂÃÅÉÈËÊÍÌÏÎÓÒÖÔÕÚÙÜÛÇÑ', 
            'aaaaaaeeeeiiiiooooouuuucnAAAAAAEEEEIIIIOOOOOUUUUCN'
          ), 
          '[^a-zA-Z0-9]', ''
        )
      )) = LOWER(TRIM(
        REGEXP_REPLACE(
          TRANSLATE(t1.Principal_address_country_code, 
            'áàäâãåéèëêíìïîóòöôõúùüûçñÁÀÄÂÃÅÉÈËÊÍÌÏÎÓÒÖÔÕÚÙÜÛÇÑ', 
            'aaaaaaeeeeiiiiooooouuuucnAAAAAAEEEEIIIIOOOOOUUUUCN'
          ), 
          '[^a-zA-Z0-9]', ''
        )
      )) THEN 3 -- special characters removed, string match
      WHEN t5.RegisteredCountryIsoCode IS NULL OR t1.Principal_address_country_code IS NULL OR TRIM(t5.RegisteredCountryIsoCode) = '' OR TRIM(t1.Principal_address_country_code) = '' THEN 2 -- empty or null
      ELSE 0 -- no match
    END AS gcobGcdsCountryOfRegistrationMatch


  -- GCOB & GCDS GCO
  , CASE
      WHEN LOWER(TRIM(t5.GlobalClientOwner)) = LOWER(TRIM(t1.gcdsGlobalCOname)) THEN 1 -- string match
      WHEN LOWER(TRIM(
        REGEXP_REPLACE(
          TRANSLATE(t5.GlobalClientOwner, 
            'áàäâãåéèëêíìïîóòöôõúùüûçñÁÀÄÂÃÅÉÈËÊÍÌÏÎÓÒÖÔÕÚÙÜÛÇÑ', 
            'aaaaaaeeeeiiiiooooouuuucnAAAAAAEEEEIIIIOOOOOUUUUCN'
          ), 
          '[^a-zA-Z0-9]', ''
        )
      )) = LOWER(TRIM(
        REGEXP_REPLACE(
          TRANSLATE(t1.gcdsGlobalCOname, 
            'áàäâãåéèëêíìïîóòöôõúùüûçñÁÀÄÂÃÅÉÈËÊÍÌÏÎÓÒÖÔÕÚÙÜÛÇÑ', 
            'aaaaaaeeeeiiiiooooouuuucnAAAAAAEEEEIIIIOOOOOUUUUCN'
          ), 
          '[^a-zA-Z0-9]', ''
        )
      )) THEN 3 -- special characters removed, string match
      WHEN t5.GlobalClientOwner IS NULL OR t1.gcdsGlobalCOname IS NULL OR TRIM(t5.GlobalClientOwner) = '' OR TRIM(t1.gcdsGlobalCOname) = '' THEN 2 -- empty or null
      ELSE 0 -- no match
    END AS gcobGcdsGlobalClientOwnerMatch


  -- GCOB & GCDS GCO location
  , CASE
      WHEN LOWER(TRIM(t5.GlobalClientOwnerLocation)) = LOWER(TRIM(t1.gcdsGlobalCOlocation)) THEN 1 -- string match
      WHEN LOWER(TRIM(
        REGEXP_REPLACE(
          TRANSLATE(t5.GlobalClientOwnerLocation, 
            'áàäâãåéèëêíìïîóòöôõúùüûçñÁÀÄÂÃÅÉÈËÊÍÌÏÎÓÒÖÔÕÚÙÜÛÇÑ', 
            'aaaaaaeeeeiiiiooooouuuucnAAAAAAEEEEIIIIOOOOOUUUUCN'
          ), 
          '[^a-zA-Z0-9]', ''
        )
      )) = LOWER(TRIM(
        REGEXP_REPLACE(
          TRANSLATE(t1.gcdsGlobalCOlocation, 
            'áàäâãåéèëêíìïîóòöôõúùüûçñÁÀÄÂÃÅÉÈËÊÍÌÏÎÓÒÖÔÕÚÙÜÛÇÑ', 
            'aaaaaaeeeeiiiiooooouuuucnAAAAAAEEEEIIIIOOOOOUUUUCN'
          ), 
          '[^a-zA-Z0-9]', ''
        )
      )) THEN 3 -- special characters removed, string match
      WHEN t5.GlobalClientOwnerLocation IS NULL OR t1.gcdsGlobalCOlocation IS NULL OR TRIM(t5.GlobalClientOwnerLocation) = '' OR TRIM(t1.gcdsGlobalCOlocation) = '' THEN 2 -- empty or null
      ELSE 0 -- no match
    END AS gcobGcdsGlobalClientOwnerLocationMatch


  -- GCOB & GCDS Next Review Date
  , CASE
      WHEN t5.nextreviewdate = t1.gcdsCDDNextReviewdate THEN 1 
      WHEN t5.nextreviewdate IS NULL OR t1.gcdsCDDNextReviewdate IS NULL OR TRIM(t5.nextreviewdate) = '' OR TRIM(t1.gcdsCDDNextReviewdate) = '' THEN 2 -- empty or null
      ELSE 0 
    END AS gcobGcdsNextReviewDateMatch


  -- GCOB & GCDS Risk Level
  , CASE
      WHEN t5.ValidatedRiskLevel = t1.gcdsCDDRating THEN 1
      WHEN t5.ValidatedRiskLevel IS NULL OR t1.gcdsCDDRating IS NULL OR TRIM(t5.ValidatedRiskLevel) = '' OR TRIM(t1.gcdsCDDRating) = '' THEN 2 -- empty or null
      ELSE 0 
    END AS gcobGcdsValidatedRiskLevelMatch
  
  -- GCOB & GCDS Business Line
  , CASE
      WHEN LOWER(TRIM(t5.BusinessLineName)) = LOWER(TRIM(t1.gcdsCObusinesslinedescription)) THEN 1 -- string match
      WHEN LOWER(TRIM(
        REGEXP_REPLACE(
          TRANSLATE(t5.BusinessLineName, 
            'áàäâãåéèëêíìïîóòöôõúùüûçñÁÀÄÂÃÅÉÈËÊÍÌÏÎÓÒÖÔÕÚÙÜÛÇÑ', 
            'aaaaaaeeeeiiiiooooouuuucnAAAAAAEEEEIIIIOOOOOUUUUCN'
          ), 
          '[^a-zA-Z0-9]', ''
        )
      )) = LOWER(TRIM(
        REGEXP_REPLACE(
          TRANSLATE(t1.gcdsCObusinesslinedescription, 
            'áàäâãåéèëêíìïîóòöôõúùüûçñÁÀÄÂÃÅÉÈËÊÍÌÏÎÓÒÖÔÕÚÙÜÛÇÑ', 
            'aaaaaaeeeeiiiiooooouuuucnAAAAAAEEEEIIIIOOOOOUUUUCN'
          ), 
          '[^a-zA-Z0-9]', ''
        )
      )) THEN 3 -- special characters removed, string match
      WHEN t5.BusinessLineName IS NULL OR t1.gcdsCObusinesslinedescription IS NULL OR TRIM(t5.BusinessLineName) = '' OR TRIM(t1.gcdsCObusinesslinedescription) = '' THEN 2 -- empty or null
      ELSE 0 -- no match
    END AS gcobGcdsBusinessLineMatch

FROM gcdsRawData t1
FULL OUTER JOIN sggGcobData t5 ON t1.gcobid = t5.UniqueGcobId

In [0]:
df_InternalUse=spark.table('GcobGcdsClientComparison')

In [0]:
save_to_saradar_storage_account(df_InternalUse, party_dataobject, radar_datamodel_version_number, environment)